# DINOv2 + YOLO + LoRA sin MLP: evaluación de test
## Fase 1. Preparación y verificación de datos
Objetivo: medir el efecto de adaptar el encoder sin MLP. Pregunta: ¿LoRA solo mejora DINO+YOLO congelado y cómo se compara con la cadena MLP+LoRA? Hipótesis: la adaptación puede mejorar recuperación, sin asumir transferencia. Intervención: adaptadores LoRA ya entrenados; controles: backbone, detector, procesador, splits y métricas fijados. Muestra: público v1 inmutable, 277 fotos/60 identidades; PetTrace v6, 502 fotos/50 identidades, galerías separadas. Semillas 45–47; selección 45 por mAP macro de validación antes de estos tests. Sin entrenamiento ni limpieza en este notebook. Se preserva evidencia local y se publica obligatoriamente en MLflow.

In [1]:
import csv, hashlib, importlib.metadata, json, os, platform, shutil, sys, tarfile, time
from pathlib import Path
from datetime import datetime, timezone
from types import SimpleNamespace
from uuid import uuid4
from dotenv import load_dotenv
import numpy as np
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"  # CPU interpolation for unsupported MPS operations.
import torch
import mlflow
from huggingface_hub import hf_hub_download, snapshot_download
from pettrace.tracking import PRODUCTION_URI, source_identity, publish_evaluation
from pettrace.data.benchmark import Dataset, Photo
from pettrace.data.refined_sources import RELEASES
from pettrace.data.verification import sha256
from pettrace.experiments.e1 import _private_dataset
from pettrace.experiments.zero_shot import save_publication_input, load_publication_input

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Python 3.11–3.13 required")
if not torch.backends.mps.is_available():
    raise RuntimeError("This fixed execution requires the Mac MPS device")
ROOT = Path.cwd()
assert (ROOT / "src/pettrace").is_dir(), "Run from the repository root"
load_dotenv(ROOT / ".env"); load_dotenv(ROOT / ".env.local")
os.environ["MLFLOW_TRACKING_URI"] = PRODUCTION_URI
os.environ["MLFLOW_REGISTRY_URI"] = PRODUCTION_URI
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
for name in ("HF_TOKEN", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    if not os.environ.get(name):
        raise RuntimeError(f"Missing {name}")
mlflow.set_tracking_uri(PRODUCTION_URI); mlflow.set_registry_uri(PRODUCTION_URI)
DATASET_COMMIT = "52bfe26384c4230008fe4707aa0cb145988766a8"
PUBLIC_RUN = "358cb901f3e34483b24acc6ce93649bb"
MODEL_REPO = "facebook/dinov2-large"
MODEL_REVISION = "47b73eefe95e8d44ec3623f8890bd894b6ea2d6c"
LOG_TO_MLFLOW = True
RUN_LABEL = "20261120-v1"  # Fixed study receipt; reruns resume publication.
os.environ["PETTRACE_EXECUTION_ID"] = RUN_LABEL
REPORT = ROOT / "reports/dino-lora-test" / RUN_LABEL
REPORT.mkdir(parents=True, exist_ok=True)
CACHE = ROOT / "reports/c1-dinov2-reference/cache"
CACHE.mkdir(exist_ok=True)
SOURCE = source_identity()
method_cells = json.loads((ROOT / "notebooks/dino_lora_test.ipynb").read_text())["cells"]
METHOD_SHA256 = hashlib.sha256(json.dumps([c["source"] for c in method_cells],sort_keys=True).encode()).hexdigest()
print("Study:", RUN_LABEL, "| Device: MPS | Package:", SOURCE["package_version"])

# Lock selection using only the already published validation scores.
TRAINING_RUNS={45:'209b96e9182c45d3a69e5671a5201e3c',46:'5e1909f4e9ae47f0a9551df350072c7c',47:'eaade411fa2141dc97137e421d405d2d'}
client=mlflow.MlflowClient(); recovered=[]
for seed,rid in TRAINING_RUNS.items():
    run=client.get_run(rid);assert run.info.status=='FINISHED'
    folder=REPORT/f'seed-{seed}';folder.mkdir(exist_ok=True)
    for artifact in ('design.json','adapter.pt','training-result.json','training-publication.json'):
        client.download_artifacts(rid,'training/'+artifact,str(folder))
    design=json.loads((folder/'design.json').read_text())
    assert design['mode']=='lora' and design['seed']==seed
    assert design['study']['model_revision']==MODEL_REVISION
    assert design['checkpoint_sha256']==sha256(folder/'adapter.pt')
    receipt=json.loads((folder/'training-publication.json').read_text())
    assert receipt['published'] and receipt['run_id']==rid
    recovered.append(dict(seed=seed,run_id=rid,recovered=str(folder),checkpoint_sha256=design['checkpoint_sha256'],metrics=run.data.metrics,design=design))
selection=max(recovered,key=lambda r:(r['metrics']['val.mAP'],-r['seed']))['seed']
assert selection==45
locked=dict(selected_lora_seed=selection,training_runs=recovered,
 scope='Retrospective LoRA-only test; all seeds fixed before this test; no test tuning.',
 registration='No registration or alias promotion; original full-study setting unchanged')
lock=REPORT/'locked-before-test.json'
if lock.exists():assert json.loads(lock.read_text())==locked
else:lock.write_text(json.dumps(locked,indent=2))

client = mlflow.MlflowClient()
receipt_path = Path(client.download_artifacts(PUBLIC_RUN, "report/dataset-selection.json", str(CACHE)))
receipt = json.loads(receipt_path.read_text())
assert receipt["source"]["commit"] == "a724c1099a5432609c853958e88b68c4cf82d871"
expected = {(r["animal"], r["species"], r["sha256"]) for r in receipt["rows"]}
photos, rows = [], []
for source in ("lcw", "dogs-world"):
    spec = RELEASES[source]
    root = Path(snapshot_download(spec["repo_id"], repo_type="dataset", revision=spec["v2_commit"],
        allow_patterns=["versions/v1/dataset.json", "versions/v1/metadata.csv", "versions/v1/data/test/*"]))
    manifest = json.loads((root / "versions/v1/dataset.json").read_text())
    meta = root / "versions/v1/metadata.csv"
    assert sha256(meta) == manifest["file_sha256"]["versions/v1/metadata.csv"]
    with meta.open(newline="") as f:
        all_rows = list(csv.DictReader(f))
    subset = [r for r in all_rows if r["split"] == "test"]
    assert not {r["identity_id"] for r in subset} & {r["identity_id"] for r in all_rows if r["split"] != "test"}
    by_hash = {r["sha256"]: r for r in subset}
    assert len(by_hash) == len(subset)
    target_dir = CACHE / "images" / source
    target_dir.mkdir(parents=True, exist_ok=True)
    found = set()
    for shard in manifest["shards"]:
        if "/data/test/" not in shard["path"]:
            continue
        assert not Path(shard["path"]).is_absolute() and ".." not in Path(shard["path"]).parts
        archive = root / shard["path"]
        assert sha256(archive) == shard["sha256"]
        with tarfile.open(archive) as tar:
            for member in tar:
                name = Path(member.name)
                assert member.isfile() and name.name == member.name
                if name.suffix == ".json":
                    continue
                assert name.suffix.lower() in {".jpg", ".jpeg", ".png"}
                assert name.stem in by_hash and name.stem not in found
                target = target_dir / name.name
                if not target.exists() or sha256(target) != name.stem:
                    data = tar.extractfile(member).read()
                    assert hashlib.sha256(data).hexdigest() == name.stem
                    target.write_bytes(data)
                row = by_hash[name.stem]
                photos.append(Photo(str(target), row["identity_id"], row["species"], "test", 0.0))
                rows.append((row["identity_id"], row["species"], name.stem))
                found.add(name.stem)
    assert found == set(by_hash)
assert set(rows) == expected and len(rows) == len(expected) == 277
identity = {"prepared_fingerprint": receipt["prepared_fingerprint"], "split": "test", "rows": sorted(rows)}
digest = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
assert digest == receipt["evaluation_fingerprint"]
public_ds = Dataset(receipt["source"]["repo"], "v2", receipt["source"]["commit"], digest, str(CACHE),
    {p.animal: {"species":p.species,"split":"test"} for p in photos}, photos)
private_ds = _private_dataset("v6", DATASET_COMMIT, CACHE / "private")
assert len(private_ds.photos) == 502 and private_ds.fingerprint == "6dcae9a47764beb0"
assert not {r[2] for r in rows} & {sha256(p.path) for p in private_ds.photos}
(REPORT / "dataset-selection.json").write_text(json.dumps(receipt, indent=2))
print("Verified:", len(public_ds.photos), "public photos /", len(public_ds.animals), "identities;",
      len(private_ds.photos), "private photos /", len(private_ds.animals), "identities")


2026/11/20 18:22:29 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/alessandro/Develop/personal/pettrace-ml/.venv/lib/python3.11/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Study: 20261120-v1 | Device: MPS | Package: 0.2.42


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Fetching 506 files:   0%|          | 0/506 [00:00<?, ?it/s]

Verified: 277 public photos / 60 identities; 502 private photos / 50 identities


## Fase 2. Representación y entrenamiento
Se recuperan los tres checkpoints terminados y se verifican hashes y procedencia. Entrenados con 193.276 fotos disponibles y 273 de validación; consultar diseño e historial publicados para exposición efectiva. No se reentrena ni se añade MLP. Se verifica paridad CPU/MPS por checkpoint antes de evaluar. Métodos externos: [DINOv2](https://arxiv.org/abs/2304.07193), [LoRA](https://arxiv.org/abs/2106.09685), [batch-hard triplet](https://arxiv.org/abs/1703.07737). El protocolo comparativo y la selección son decisiones del estudio.

In [2]:
from pettrace.training.dinov2 import build, Config, Pipeline
from pettrace.training import lora
from pettrace.evaluation.core import load_model
from pettrace.models.artifacts import Model
from pettrace.models.golden import golden_images, compare, fingerprint
model_dir=Path(snapshot_download(MODEL_REPO,revision=MODEL_REVISION,allow_patterns=['model.safetensors','config.json','preprocessor_config.json']))
base=load_model('models:/pettrace-embedder/2');assert base.golden_ok
hashes={name:sha256(model_dir/name) for name in ('model.safetensors','config.json','preprocessor_config.json')}
hashes.update({'detector.'+name:value for name,value in base.sha256.items() if 'detector' in name})
for r in recovered:assert hashes==r['design']['base_hashes']
torch.set_num_threads(4)
print('Three LoRA-only checkpoints and base hashes verified. Selected seed:',selection)


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Three LoRA-only checkpoints and base hashes verified. Selected seed: 45


## Fase 3. Evaluación y análisis
Recuperación cerrada de misma especie; leave-one-out por fotografía. Rank-1/5 por identidad máxima, mAP por fotos, EER descriptiva sin calibración operacional. Se evalúan todas las semillas en ambos dominios y se publican los seis resultados, incluidos negativos. Comparación emparejada contra E2 congelado, bootstrap descriptivo por identidad con galería y checkpoint fijos. Resultados en las salidas y reportes; no se anticipan. Limitaciones: análisis retrospectivo, tests previamente observados, sin separación acreditada entre sesiones; los intervalos no incluyen variabilidad de entrenamiento. Registro y promoción quedan fuera, preservando la configuración original del estudio completo.

In [3]:
from pettrace.evaluation.core import evaluate
from pettrace.evaluation.figures import save
from pettrace.evaluation.paired import paired_comparison
publications={}
summary=[]
for r in sorted(recovered,key=lambda r:r['seed']):
    seed=r['seed'];checkpoint=Path(r['recovered'])/'adapter.pt'
    encoder,processor=build('lora','cpu',Config(**r['design']['config']))
    saved=lora.load_adapter(encoder,checkpoint)
    assert saved['config']==r['design']['config']
    projected=Pipeline(encoder,processor,base.pipeline.detector)
    golden_cpu=projected.embed(golden_images())
    encoder.to('mps');got=projected.embed(golden_images())
    ok,min_cos,max_abs=compare(golden_cpu,got);assert ok,(min_cos,max_abs)
    parity=dict(cpu_mps_min_cosine=min_cos,cpu_mps_max_abs=max_abs)
    (REPORT/'parity.json').write_text(json.dumps(parity,indent=2))
    adapted_hashes={**hashes,'adapter.pt':sha256(checkpoint)}
    adapted=Model(f"runs:/{r['run_id']}/training/adapter.pt",str(checkpoint.parent),projected,
        f'dinov2-lora-only-seed{seed}','',adapted_hashes,True,fingerprint(got))
    for domain,dataset in (('public',public_ds),('private',private_ds)):
        folder=REPORT/f'seed-{seed}'/domain;folder.mkdir(parents=True,exist_ok=True)
        for name in ('locked-before-test.json','dataset-selection.json','parity.json'):
            shutil.copy2(REPORT/name,folder/name)
        if (folder/'evaluation-input.json').exists():
            ev=load_publication_input(folder);ev.dataset=dataset
        else:
            started=time.monotonic();ev=evaluate(dataset,projected,zero_shot=True)
            save(ev,adapted,str(folder));save_publication_input(ev,folder)
            (folder/'runtime.json').write_text(json.dumps(dict(**SOURCE,execution_id=RUN_LABEL,
                inference_backend='torch_mps_lora',seconds=time.monotonic()-started,
                model_sha256=adapted_hashes,notebook_method_sha256=METHOD_SHA256),indent=2))
        baseline=json.loads((ROOT/'reports/c1-dinov2-reference/20261118T063806Z-cc261f74'/domain/'evaluation-input.json').read_text())
        assert baseline['dataset']['fingerprint']==dataset.fingerprint
        paired=paired_comparison(baseline['result'],ev.result)
        (folder/'paired-vs-c1.json').write_text(json.dumps(paired,indent=2))
        run=publish_evaluation(ev,adapted,folder,SOURCE['source_sha256'],experiment='DINO-LoRA-only-test',
            expected_model_uri=adapted.uri,expected_model_sha256=adapted_hashes,
            expected_dataset_version=dataset.version,expected_protocol='zero-shot')
        assert client.get_run(run).info.status=='FINISHED'
        receipt=dict(published=True,run_id=run,training_run=r['run_id'],seed=seed,domain=domain)
        (folder/'publication.json').write_text(json.dumps(receipt,indent=2))
        client.log_artifact(run,str(folder/'publication.json'),'report')
        downloaded=client.download_artifacts(run,'report/paired-vs-c1.json',str(folder/'verified'))
        assert sha256(downloaded)==sha256(folder/'paired-vs-c1.json')
        publications[f'{seed}-{domain}']=receipt
        summary.append(dict(seed=seed,domain=domain,metrics=ev.result['test']['global'],paired=paired,run_id=run))
        (REPORT/'summary.json').write_text(json.dumps(summary,indent=2))
        (REPORT/'publication.json').write_text(json.dumps(publications,indent=2))
        print(seed,domain,ev.result['test']['global'],'net hits',paired['net_hits'],run,flush=True)
    del projected,encoder
    import gc
    gc.collect();torch.mps.empty_cache()
assert len(publications)==6
print('COMPLETE: six LoRA-only test evaluations published; no registration or alias changes.')


/Users/alessandro/Develop/personal/pettrace-ml/.venv/lib/python3.11/site-packages/torch/nn/functional.py:4594: UserWarning: The operator 'aten::upsample_bicubic2d.out' is not currently supported on the MPS backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/mps/MPSFallback.mm:13.)
  return torch._C._nn.upsample_bicubic2d(


🏃 View run DINO-LoRA-only-test · dinov2-lora-only-seed45 on lcw+dow-refined@v2 at: https://ml.pettrace.app/#/experiments/3/runs/ab0b0aaf98e24ae18737c7f41da42d03
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3


45 public {'queries': 277, 'gallery_animals': 34, 'rank1': 0.9602888086642599, 'rank2': 0.9963898916967509, 'rank5': 1.0, 'rank10': 1.0, 'mAP': 0.913996008188436} net hits 11 ab0b0aaf98e24ae18737c7f41da42d03


🏃 View run DINO-LoRA-only-test · dinov2-lora-only-seed45 on pettrace-reid@v6 at: https://ml.pettrace.app/#/experiments/3/runs/34150624a2d4450b9b61e2cf8dac314b
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3


45 private {'queries': 502, 'gallery_animals': 33, 'rank1': 0.9581673306772909, 'rank2': 0.9920318725099602, 'rank5': 1.0, 'rank10': 1.0, 'mAP': 0.9039823933162123} net hits 5 34150624a2d4450b9b61e2cf8dac314b


🏃 View run DINO-LoRA-only-test · dinov2-lora-only-seed46 on lcw+dow-refined@v2 at: https://ml.pettrace.app/#/experiments/3/runs/16214e8f0e0f4740a0bd2ded59ab3626
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3


... (4 outputs left out)

47 public {'queries': 277, 'gallery_animals': 34, 'rank1': 0.9602888086642599, 'rank2': 0.9963898916967509, 'rank5': 1.0, 'rank10': 1.0, 'mAP': 0.915829976272452} net hits 11 fa9bd58fb945454bb31a8f768dac9b46


🏃 View run DINO-LoRA-only-test · dinov2-lora-only-seed47 on pettrace-reid@v6 at: https://ml.pettrace.app/#/experiments/3/runs/ec35e029afdb4e6aa13ffa8a703a9e10
🧪 View experiment at: https://ml.pettrace.app/#/experiments/3


47 private {'queries': 502, 'gallery_animals': 33, 'rank1': 0.9621513944223108, 'rank2': 0.9920318725099602, 'rank5': 1.0, 'rank10': 1.0, 'mAP': 0.903661951592782} net hits 7 ec35e029afdb4e6aa13ffa8a703a9e10


COMPLETE: six LoRA-only test evaluations published; no registration or alias changes.
